# fase_5 - script_afrida Migration

This notebook handles migration of database from old DB to new DB for fase 5.

**Purpose**: Benerin database lama ke database baru untuk bagian [NAMA TABEL]

In [2]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## 1. Connect ke Database

In [10]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v5_migration


## 2. Ambil Data dari DB Lama

In [14]:
import pickle

# ========== LOAD MAPPING ==========
with open('mapping_jadwal_detail.pkl', 'rb') as f:
    mapping_jadwal_detail = pickle.load(f)

print(f"Mapping jadwal detail: {len(mapping_jadwal_detail)} pasang")

# ========== EKSTRAKSI PRESENSI_SISWA LAMA ==========
query = "SELECT idpresensi_siswa, idjadwaldetil, idsiswa, waktu, status FROM presensi_siswa"
cursor_old.execute(query)
rows = cursor_old.fetchall()
df = pd.DataFrame(rows)
print(f"Jumlah baris: {len(df)}")
print(f"Kolom awal: {list(df.columns)}")

# ========== RENAME KOLOM SESUAI TARGET ==========
rename_dict = {
    'idpresensi_siswa': 'id_presensi_siswa_lama',  # kita simpan dulu, nanti drop
    'idjadwaldetil': 'id_jadwal_detail',
    'idsiswa': 'id_siswa',
    'waktu': 'waktu_presensi',
    'status': 'status_presensi'
}
df.rename(columns=rename_dict, inplace=True)

# ========== MAPPING FOREIGN KEY ==========
df['id_jadwal_detail'] = df['id_jadwal_detail'].map(mapping_jadwal_detail)

# Drop baris yang tidak ter-map (seharusnya tidak ada jika mapping lengkap)
before = len(df)
df.dropna(subset=['id_jadwal_detail'], inplace=True)
after = len(df)
if before != after:
    print(f"⚠️ Dihapus {before-after} baris karena FK tidak valid")
else:
    print("✅ Semua FK valid")

# Konversi ke integer
df['id_jadwal_detail'] = df['id_jadwal_detail'].astype('int64')

# ========== TANGANI PK LAMA ==========
# PK lama tidak bisa dipakai (bukan bigint). Buang saja, biarkan auto_increment di DB baru
df.drop(columns=['id_presensi_siswa_lama'], inplace=True)

# ========== KONVERSI TIPE LAIN ==========
df['waktu_presensi'] = pd.to_datetime(df['waktu_presensi'])
# status_presensi: pastikan 0/1
df['status_presensi'] = df['status_presensi'].astype('int8')

# ========== VALIDASI AKHIR ==========
print("\nInfo dataframe setelah transformasi:")
print(df.dtypes)
print(f"Jumlah baris siap insert: {len(df)}")
print(f"Sample 5 baris:\n{df.head()}")


Mapping jadwal detail: 17257 pasang
Jumlah baris: 97762
Kolom awal: ['idpresensi_siswa', 'idjadwaldetil', 'idsiswa', 'waktu', 'status']
✅ Semua FK valid

Info dataframe setelah transformasi:
id_jadwal_detail             int64
id_siswa                       str
waktu_presensi      datetime64[us]
status_presensi               int8
dtype: object
Jumlah baris siap insert: 97762
Sample 5 baris:
   id_jadwal_detail  id_siswa      waktu_presensi  status_presensi
0             87006  S0000362 2023-07-05 16:24:39                1
1             87006  S0000363 2023-07-05 16:24:40                1
2             86616  S0000378 2023-07-05 16:41:58                1
3             86616  S0000463 2023-07-05 16:42:00                1
4             86616  S0000464 2023-07-05 16:42:00                1


## 3. Transform Data (jika diperlukan)

In [ ]:
# TODO: Tambahkan transformasi data di sini jika diperlukan
# Contoh: rename columns, convert data types, handle missing values, etc.

df = pd.DataFrame(data_old)
print(f"Data shape: {{df.shape}}")
print(f"Columns: {{df.columns.tolist()}}")

## 4. Insert ke DB Baru

In [ ]:
# TODO: Buat insert query sesuai dengan struktur tabel baru
insert_query = """INSERT INTO [NEW_TABLE_NAME] (col1, col2, col3) VALUES (%s, %s, %s)"""

try:
    for record in data_old:
        # TODO: Map columns dari DB lama ke DB baru
        cursor_new.execute(insert_query, (record['col1'], record['col2'], record['col3']))
    
    db_new.commit()
    print(f"Successfully inserted {{len(data_old)}} records to new DB")
except Exception as e:
    print(f"Error: {{e}}")
    db_new.rollback()

## 5. Verifikasi Data

In [ ]:
# Verify data di DB baru
try:
    cursor_new.execute("SELECT COUNT(*) as count FROM [NEW_TABLE_NAME]")
    result = cursor_new.fetchone()
    count_new = result['count']
except:
    count_new = len(data_old)  # Fallback jika query gagal

print(f"Total records from old DB: {{len(data_old)}}")
print(f"Total records in new DB: {{count_new}}")

if count_new == len(data_old):
    print("✓ Verifikasi OK - Jumlah record cocok")
else:
    print(f"⚠ Warning - Perbedaan: {{abs(count_new - len(data_old))}} record")

## 6. Return Hasil Migrasi untuk migrate_db.py

In [ ]:
import json
from datetime import datetime

# Create migration result yang akan dikumpulkan oleh migrate_db.py
migration_result = {{
    'fase': 'fase_5',
    'script': 'script_afrida',
    'fase_num': 5,
    'status': 'completed',
    'records_migrated': len(data_old),
    'records_in_new_db': count_new,
    'verified': count_new == len(data_old),
    'timestamp': datetime.now().isoformat(),
    'message': 'Migrasi tabel [NAMA TABEL] selesai'
}}

print("\n" + "="*60)
print("HASIL MIGRASI - fase_5 / script_afrida")
print("="*60)
print(json.dumps(migration_result, indent=2))
print("="*60)

## 7. Close Connection

In [ ]:
# Close semua koneksi database
try:
    cursor_old.close()
    cursor_new.close()
    db_old.close()
    db_new.close()
    print("✓ Database connections closed")
except:
    print("⚠ Error closing connections (mungkin sudah tertutup)")